# 10 — Retrieval Strategies (Milestone M4)

**DSML stage:** modeling (retrieval). Implements and compares the three retrieval modes from the
feasibility studies on the Nvidia PoC graph:

| | Strategy | Best for | Weakness |
|---|---|---|---|
| A | **Text-to-Cypher** | rigid quantitative questions | brittle syntax, schema drift |
| B | **Vector RAG** | isolated semantic lookups | no structure, misses multi-hop |
| C | **Entity-first hybrid** *(recommended)* | supply-chain / risk intelligence | needs entity anchors in the query |

Strategy C: detect anchor entities in the query (canonical alias matching — deterministic, no LLM),
pull their 1–2-hop relation subgraph, then run vector search **scoped to EvidenceSpans that mention the
anchors** — structure + semantics combined.

In [ ]:
import json
import os
import re
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from neo4j import GraphDatabase
from sentence_transformers import SentenceTransformer
from litellm import completion

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(PROJECT_ROOT / ".env")

driver = GraphDatabase.driver(os.environ["NEO4J_URI"], auth=(os.environ["NEO4J_USER"], os.environ["NEO4J_PASSWORD"]))
driver.verify_connectivity()
model = SentenceTransformer(os.getenv("EMBEDDING_MODEL", "Qwen/Qwen3-Embedding-0.6B"))
LLM_MODEL = os.environ["LLM_MODEL"]
CANONICAL = json.loads((PROJECT_ROOT / "artifacts/canonical_entities.json").read_text())

# Queries get a model-appropriate prompt; passages were embedded plain (notebook 09).
# Qwen3-Embedding ships a built-in 'query' prompt; bge-style models need a manual prefix.
BGE_QUERY_PREFIX = "Represent this sentence for searching relevant passages: "

def embed_query(q: str) -> list[float]:
    if "query" in (model.prompts or {}):
        return model.encode([q], prompt_name="query", normalize_embeddings=True)[0].tolist()
    return model.encode([BGE_QUERY_PREFIX + q], normalize_embeddings=True)[0].tolist()

def run_cypher(query: str, **params) -> list[dict]:
    with driver.session() as session:
        return [dict(r) for r in session.run(query, **params)]
print("ready")

## A. Text-to-Cypher — LLM writes the query from a schema card

In [ ]:
SCHEMA_CARD = """Nodes:
  Company(cik INT UNIQUE, ticker, name, tier, sec_filer)
  Filing(accession_no UNIQUE, form, filing_date DATE, url)
  FilingSection(section_key UNIQUE, section_id, title)
  RiskFactor(risk_id UNIQUE, summary, category)
  Metric(metric_id UNIQUE, metric ['revenue','capex','rnd','net_income'], value FLOAT, unit, period_start DATE, period_end DATE)
  Product(name UNIQUE, type)
  EvidenceSpan(chunk_id UNIQUE, text, source_url)
Relationships:
  (Company)-[:FILED {date}]->(Filing)
  (Filing)-[:HAS_SECTION]->(FilingSection)
  (EvidenceSpan)-[:FROM_SECTION]->(FilingSection)
  (EvidenceSpan)-[:MENTIONS]->(Company)
  (Company)-[:REPORTS_METRIC {accession_no}]->(Metric)
  (Company)-[:DISCLOSES_RISK {start_date, status}]->(RiskFactor)
  (RiskFactor)-[:HAS_EVIDENCE {quote}]->(EvidenceSpan)
  (Company)-[:SUPPLIES_TO|DEPENDS_ON|CUSTOMER_OF|COMPETES_WITH {start_date, status, evidence_chunk_ids, evidence_quote}]->(Company)
  (Product)-[:MENTIONED_IN]->(EvidenceSpan)"""

def text_to_cypher(question: str) -> tuple[str, list[dict]]:
    prompt = (
        f"Write a single read-only Cypher query for Neo4j 5 answering the question below.\n"
        f"Schema:\n{SCHEMA_CARD}\n\nQuestion: {question}\n\n"
        f"Rules: RETURN descriptive column aliases; LIMIT 25; no writes; reply with ONLY the Cypher, no fences."
    )
    resp = completion(model=LLM_MODEL, messages=[{"role": "user", "content": prompt}], max_tokens=400, temperature=0.0)
    cypher = re.sub(r"^```(cypher)?|```$", "", resp.choices[0].message.content.strip(), flags=re.MULTILINE).strip()
    try:
        return cypher, run_cypher(cypher)
    except Exception as e:
        return cypher, [{"error": str(e)[:200]}]

cypher, rows = text_to_cypher("What was Nvidia's revenue for the fiscal year ending 2024-01-28?")
print(cypher, "\n")
pd.DataFrame(rows)

## B. Plain vector RAG over EvidenceSpans

In [ ]:
def vector_rag(question: str, k: int = 5) -> pd.DataFrame:
    rows = run_cypher(
        """CALL db.index.vector.queryNodes('evidence_embedding', $k, $vec) YIELD node, score
        RETURN node.chunk_id AS chunk_id, score, node.sub_heading AS sub_heading,
               left(node.text, 180) AS preview""",
        k=k, vec=embed_query(question),
    )
    return pd.DataFrame(rows)

vector_rag("How do export controls affect Nvidia's business in China?")

## C. Entity-first hybrid (recommended) — anchors → subgraph → scoped vectors

In [ ]:
ALIAS_TO_ID = {}
for name, spec in CANONICAL.items():
    for alias in {name, *spec["aliases"]}:
        ALIAS_TO_ID[alias.lower()] = (name, spec["entity_id"])
ALIAS_RES = [(re.compile(rf"\b{re.escape(a)}\b", re.I), v) for a, v in ALIAS_TO_ID.items()]

def detect_anchors(question: str) -> dict[str, int]:
    return {name: eid for pat, (name, eid) in ALIAS_RES if pat.search(question)}

def hybrid_retrieve(question: str, k_chunks: int = 6, hops: int = 2) -> dict:
    anchors = detect_anchors(question)
    anchor_ids = list(anchors.values()) or [1045810]  # default anchor: Nvidia (the PoC filer)
    edges = run_cypher(
        f"""MATCH (a:Company) WHERE a.cik IN $ids
        MATCH p = (a)-[r:SUPPLIES_TO|DEPENDS_ON|CUSTOMER_OF|COMPETES_WITH*1..{hops}]-(b:Company)
        UNWIND relationships(p) AS rel
        RETURN DISTINCT startNode(rel).name AS source, type(rel) AS relation, endNode(rel).name AS target,
               rel.status AS status, rel.evidence_quote AS quote, rel.evidence_chunk_ids AS chunk_ids""",
        ids=anchor_ids,
    )
    risks = run_cypher(
        """MATCH (a:Company)-[d:DISCLOSES_RISK {status:'Active'}]->(rf:RiskFactor)
        WHERE a.cik IN $ids
        CALL db.index.vector.queryNodes('risk_embedding', 25, $vec) YIELD node, score
        WITH rf, node, score WHERE node = rf
        RETURN rf.summary AS summary, rf.category AS category, score ORDER BY score DESC LIMIT 6""",
        ids=anchor_ids, vec=embed_query(question),
    )
    chunks = run_cypher(
        """CALL db.index.vector.queryNodes('evidence_embedding', 40, $vec) YIELD node, score
        MATCH (node)-[:MENTIONS]->(c:Company) WHERE c.cik IN $ids
        RETURN DISTINCT node.chunk_id AS chunk_id, score, node.text AS text,
               node.source_url AS source_url ORDER BY score DESC LIMIT $k""",
        ids=anchor_ids, vec=embed_query(question), k=k_chunks,
    )
    return {"anchors": anchors, "edges": edges, "risks": risks, "chunks": chunks}

result = hybrid_retrieve("How does Nvidia depend on TSMC, and what supply-chain risks does it disclose?")
print("anchors:", result["anchors"])
print(f"{len(result['edges'])} subgraph edges, {len(result['risks'])} scoped risks, {len(result['chunks'])} scoped chunks")
pd.DataFrame(result["edges"])[["source", "relation", "target", "quote"]].head(10)

## Side-by-side comparison on the feasibility studies' retrieval flows (Nvidia-scope subset)

In [ ]:
TEST_QUERIES = [
    "Which companies does Nvidia depend on for manufacturing?",              # flow A analogue
    "How has Nvidia's AI-related risk disclosure characterized export controls?",  # flow B/F analogue
    "What evidence supports the claim that AI demand is increasing data center revenue?",  # flow G
    "Does Nvidia disclose dependency on third-party foundries?",             # flow H
]

comparison = []
for q in TEST_QUERIES:
    h = hybrid_retrieve(q)
    v = vector_rag(q, k=6)
    comparison.append({
        "question": q,
        "anchors": ", ".join(h["anchors"]) or "(default NVDA)",
        "hybrid_edges": len(h["edges"]),
        "hybrid_chunks": len(h["chunks"]),
        "vector_only_chunks": len(v),
    })
pd.DataFrame(comparison)

In [ ]:
# --- M4 (retrieval) assertion cell ---
r = hybrid_retrieve("How does Nvidia depend on TSMC?")
assert set(r["anchors"]) >= {"Nvidia", "TSMC"}, f"anchor detection failed: {r['anchors']}"
assert len(r["edges"]) >= 1, "no subgraph edges for Nvidia-TSMC"
assert any("TSMC" in (e["source"], e["target"]) for e in r["edges"])
assert len(r["chunks"]) >= 3 and all(c["source_url"] for c in r["chunks"])
assert len(vector_rag("export controls China")) == 5
driver.close()
print("M4 (retrieval) OK — all three strategies functional; hybrid returns structure + scoped evidence")